# Chapter 42. Recommender Systems & Ranking

*The runnable half of this chapter.* Every code cell below is the chapter's own code, and the words around it are the chapter's own explanation. Run the cells in order: each one uses names made by the cells before it.

Riverstone Supplies is the single worked example throughout the book, so the data here is the same data you meet in every other chapter.

Built from `manuscript/ch42-recommender-systems-and-ranking.md`.

# Chapter 42. Recommender Systems & Ranking

*Part 4 — Machine Learning & Data Science*

> **Chapter at a glance**
>
> **You will learn to:** build a popularity baseline, and know why it's hard to beat with little data · build item-based collaborative filtering by hand and in scikit-learn · evaluate a ranked list with hit rate@k, precision@k, recall@k, NDCG@k and MRR, not accuracy · understand matrix factorization by hand, then fit it with scikit-learn's `TruncatedSVD` and the `implicit` library's ALS · build a content-based recommender from item descriptions, reusing Chapter 41's TF-IDF and cosine similarity · combine methods into a hybrid recommender · handle the cold-start problem for new products and new customers · apply all of this to B2B cross-selling, and test a segment-level baseline against every model.
>
> **Before you start:** Chapter 35 (dot products, cosine similarity, PCA), Chapter 36 (baselines, the log transform, sparse matrices, train/test discipline), Chapter 37 (the accounts data, least squares, the ridge penalty, bias and variance), Chapter 38 (the basket data and market basket rules), Chapter 39 (precision, recall, average precision, the top-N table), Chapter 41 (TF-IDF, NMF). This chapter turns those pieces into personalized recommendations.
>
> **Time needed:** 8–10 hours over one week. Take it in two sittings: sections 42.0 to 42.3 (the data, collaborative filtering and how to score a ranked list), then sections 42.4 to 42.7 (matrix factorization, content, cold start and the business case).
>
> **Tools:** Python 3 with pandas and scikit-learn (installed in Chapter 35), plus the `implicit` library, installed in section 42.0 (all free).
>
> **Practice data:** Riverstone's order-line data from Chapter 38 (34,013 orders, 4,516 accounts, 24 products) and the accounts data from Chapter 37, with a short product description added to each catalog item for this chapter. Every number in this chapter was calculated, and every output shown is real.

---

## Why this matters

Chapter 38's market basket rules answered one question well: *which products tend to appear in the same order?* That's useful for a "frequently bought together" prompt on a product page, but it's the same answer for every customer. A recommender system answers a sharper question: *what should we suggest to this specific customer, right now?*

- Riverstone's account managers want a short list per customer: "these three products this account hasn't bought yet, that accounts like them buy a lot."
- A new product launches with zero sales history. Nobody has bought it yet, so nothing knows to recommend it — except its description.
- A brand-new account places its first order. There's no purchase history to learn from at all.

Those three situations need three different tools, and knowing which to reach for, and how to tell whether any of them actually works, is what this chapter teaches.

---

## In plain English

**Think of an experienced Riverstone sales rep who's worked every territory for ten years.**

Ask what to suggest to a customer who's never ordered before, and the rep falls back on "well, everyone buys the airtight seal packs" — that's a **popularity baseline**. Ask about a customer who buys a garden chair, and the rep says "show them the matching table, they're clearly in the same range" — that's **content-based filtering**, reasoning from what the product *is*. Ask about a customer who buys a lot like another customer three towns over, and the rep says "that other account also picked up dolly wheels after their pallet boxes, worth a try" — that's **collaborative filtering**, reasoning from what *similar customers* did, with no idea what a dolly wheel actually looks like.

A rep who's really good does both at once, weighing them: mostly go with what similar customers bought, but lean on the product description when a customer or a product is too new to have much history. That blend is a **hybrid recommender**, and knowing when the rep's gut feeling is actually working, rather than just sounding confident, is **evaluation**.

---

## 42.0 Setting up

### The data

This chapter's code runs from the folder `companion/ch42/`. It reads Chapter 38's order baskets (`companion/baskets/`) and Chapter 37's accounts (`companion/accounts/`), and one small helper file of its own, `products_text.py`, which sits in `companion/ch42/`. If `companion/baskets/order_lines.csv` or `companion/accounts/accounts.csv` is missing, rebuild both with the two generator commands from Chapter 38, section 38.0 (accounts first, then baskets); both generators are seeded, so you get exactly the same files.

### One new library

Section 42.4 uses **implicit**, a library built for recommenders that learn from purchases and clicks. No earlier chapter installed it. In a terminal, go to the chapter's folder, check that the prompt starts with `(.venv)` (Chapter 17, section 17.0), then install it and record it in `requirements.txt`:

**The chapter's output:**

```
# terminal
$ cd companion/ch42
$ python -m pip install implicit
$ python -m pip freeze > requirements.txt
```

- **`cd companion/ch42`** moves into the chapter's folder, so paths such as `../baskets/order_lines.csv` (one folder up, then into `baskets`) find the data.
- **`python -m pip install implicit`** downloads the library into the active environment. It ends with a line starting `Successfully installed`.
- **`python -m pip freeze > requirements.txt`** rewrites the list of installed packages, as in Chapter 17, so the new one is recorded.

`implicit` is partly written in C++, a compiled language, so pip needs a ready-made package for your system, called a **wheel**. Version 0.7.3 has wheels for Python 3.9 to 3.14 on Windows, macOS and Linux (checked on PyPI on 29 September 2026). If pip instead starts compiling and stops with an error that mentions a compiler, your Python version has no wheel: check it with `python --version`, and use a version from that list.

Start Jupyter from this folder, create a notebook called `ch42.ipynb`, and choose the `.venv` kernel. Every cell in this chapter runs in that notebook, in order. The first cell checks the install:

In [ ]:
import implicit
import sklearn

print("implicit", implicit.__version__, "| scikit-learn", sklearn.__version__)

> **Not run here.** This cell needs something this machine does not have: a database, an API key, a running service, or command-line arguments. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
implicit 0.7.3 | scikit-learn 1.9.1
```

- `import implicit` loads the new library. A `ModuleNotFoundError` here means the install went into a different environment: check which kernel the notebook is using (Chapter 17, section 17.0).
- `__version__` shows which version you have. This chapter's outputs were produced with the versions shown.

---

## 42.1 The data, and a popularity baseline

### The catalog

Chapter 38's `products.csv` has an ID, a name, a category and a price for each of the 24 products. Section 42.5 also needs a short text description of each one, so the helper file `products_text.py` adds them. It holds one Python dictionary, `DESCRIPTIONS`, that maps each product ID to a sentence, and one function, `load_products()`, that reads `products.csv` and adds a `description` column from that dictionary. Open the file in VS Code to read all 24 descriptions; here are six that come up again later:

In [2]:
import numpy as np
import pandas as pd

from products_text import load_products

products = load_products()
names = products.set_index("product_id")["product_name"]
descriptions = products.set_index("product_id")["description"]
for product_id in ["P01", "P02", "P03", "P04", "P07", "P08"]:
    print(f"{product_id} {names[product_id]}")
    print(f"    {descriptions[product_id]}")

P01 Storage Crate 50L
    Stackable 50 litre plastic storage crate for retail and home organisation, ventilated sides, food safe.
P02 Storage Crate 80L
    Larger 80 litre stackable plastic storage crate, heavy duty base, food safe, ventilated sides.
P03 Crate Lid (50L)
    Snap-fit lid for the 50 litre storage crate, dust and pest resistant, stackable when closed.
P04 Crate Lid (80L)
    Snap-fit lid for the 80 litre storage crate, dust and pest resistant, stackable when closed.
P07 Food Container 2L
    2 litre airtight food storage container, microwave and freezer safe, BPA free plastic.
P08 Food Container 5L
    5 litre airtight food storage container, microwave and freezer safe, BPA free plastic.


- `from products_text import load_products` imports the helper function from `products_text.py`. A `.py` file is a module (Chapter 17, section 17.11), and a notebook can import one that sits in its own folder, as Chapter 37 did with `lead_data.py`.
- `names` and `descriptions` are two Series indexed by product ID, so `names["P01"]` gives `'Storage Crate 50L'`. The chapter's outputs use `names` to turn IDs into names.

### Who bought what

Next, the order lines, with each line's product and account segment attached, exactly as in Chapter 38, section 38.9:

In [3]:
lines = pd.read_csv("../baskets/order_lines.csv", parse_dates=["order_date"])
accounts = pd.read_csv("../accounts/accounts.csv")
lines = lines.merge(
    products[["product_id", "product_name", "category"]], on="product_id"
)
lines = lines.merge(accounts[["account_id", "segment"]], on="account_id")
print(
    f"{len(lines):,} order lines, {lines['order_id'].nunique():,} orders, "
    f"{lines['account_id'].nunique():,} accounts, "
    f"{lines['product_id'].nunique()} products"
)

92,337 order lines, 34,013 orders, 4,516 accounts, 24 products


- `parse_dates=["order_date"]` reads the date column as dates rather than text (Chapter 18, section 18.9); exercise 12 sorts by it.
- The two `merge` calls attach each line's product name and category, and its account's segment, exactly as in Chapter 38. `len(lines)` counts the lines, and `nunique()` counts the distinct orders, accounts and products.

A recommender doesn't care about individual orders; it needs one number per account and product: *how much of this product has this account bought?* That table is the **interaction matrix**:

In [4]:
interactions = (
    lines.groupby(["account_id", "product_id"])["quantity"].sum().reset_index()
)
matrix = interactions.pivot(
    index="account_id", columns="product_id", values="quantity"
).fillna(0)
print(f"account x product matrix: {matrix.shape[0]:,} rows, {matrix.shape[1]} columns")
print(matrix.iloc[:3, :6])

account x product matrix: 4,516 rows, 24 columns
product_id  P01  P02   P03  P04  P05   P06
account_id                                
5002        4.0  0.0   0.0  0.0  0.0   0.0
5003        0.0  0.0  53.0  0.0  0.0   0.0
5004        0.0  0.0   0.0  0.0  0.0  10.0


- `groupby([...])["quantity"].sum()` adds up each account's quantity of each product across all its orders (Chapter 18, section 18.6); `reset_index()` turns the result back into a plain table with three columns.
- `pivot(index=..., columns=..., values=...)` reshapes it: one row per account, one column per product, the quantity in each cell (Chapter 18, section 18.8).
- `fillna(0)`: a pair that never appears in the orders has no value after the pivot, a blank. A missing pair means the account never bought that product, so 0 is the honest value.
- `matrix.iloc[:3, :6]` shows the first three accounts and six products: most cells are 0.

How full is the matrix?

In [5]:
bought = matrix > 0
density = bought.mean().mean()
print(f"density  (share of cells that are non-zero): {density:.1%}")
print(f"sparsity (share of cells that are zero):     {1 - density:.1%}")

density  (share of cells that are non-zero): 38.6%
sparsity (share of cells that are zero):     61.4%


- `matrix > 0` gives a True/False table: True where the account bought the product at all.
- The first `.mean()` averages each column (the mean of True/False is the share of Trues), giving one share per product; the second `.mean()` averages those 24 shares. Every column has the same number of rows, so that is the share of all cells.
- **Density** is the share of cells that are filled; **sparsity** is the opposite, the share that are empty.

Riverstone's 24-product catalog is deliberately small for this chapter, which makes every number easy to check by hand; real retail catalogs run to thousands or millions of items. The interaction matrix here is 39% full, unusually dense for a recommender problem: most catalogs are well under 1% dense (sparsity above 99%), since no customer buys more than a sliver of what's on offer. Riverstone's density comes from having only 24 products; hold onto that fact, because a couple of results below only make sense in light of it.

### The simplest recommender

The simplest possible recommender needs no personalization at all: suggest what most accounts buy.

In [6]:
popularity = bought.sum(axis=0).sort_values(ascending=False)
popularity_share = (popularity / matrix.shape[0]).rename(index=names)
print("most popular products (share of accounts that bought them):")
print((popularity_share.head(5) * 100).round(1).astype(str).add("%").to_string())
print("\nleast popular:")
print((popularity_share.tail(3) * 100).round(1).astype(str).add("%").to_string())

most popular products (share of accounts that bought them):
product_id
Airtight Seal Pack    64.6%
Crate Lid (80L)       56.3%
Crate Lid (50L)       55.4%
Stacking Bin Large    51.9%
Food Container 2L     47.6%

least popular:
product_id
Crate Trolley            23.7%
Drum 60L                 23.7%
Industrial Crate 200L    23.6%


- `bought.sum(axis=0)` counts the Trues down each column: how many accounts bought each product.
- `.rename(index=names)` swaps each product ID for its name, using the `names` Series as a lookup.
- The last part turns each share into text such as `64.6%`: multiply by 100, round, convert to text (`astype(str)`), and add a `%` sign.

**Reading it.** The Airtight Seal Pack is bought by 65% of accounts, the least popular industrial items by under a quarter. **Popularity is the seasonal naive of recommenders** (Chapter 40, section 40.3): cheap, needs no history, and a real baseline every fancier method must beat. It recommends the same thing to everyone, which is exactly its weakness: a wholesale drum buyer gets pointed at food-storage seal packs and crate lids just because they're popular overall.

---

## 42.2 Item-based collaborative filtering

### The idea, by hand

**Collaborative filtering** finds patterns in *who bought what*, with no knowledge of what the products are. **Item-based** filtering asks: which products are bought by similar sets of customers? Two products are similar if the customers who buy one tend to also buy the other, measured with cosine similarity on the columns of the interaction matrix. This is exactly Chapter 35's cosine similarity (section 35.2), now comparing products by their customers instead of customers by their products.

In [7]:
toy = pd.DataFrame(
    {
        "Storage Crate": [3, 4, 0, 0, 8],
        "Food Container": [0, 0, 6, 5, 0],
        "Crate Lid": [5, 3, 0, 0, 10],
        "Chopping Board": [0, 0, 2, 3, 1],
    },
    index=[
        "Sharma Hardware",
        "Metro Mart",
        "Coastal Foods",
        "Royal Banquets",
        "Prime Wholesale",
    ],
)
print(toy)

crate = toy["Storage Crate"].to_numpy()
lid = toy["Crate Lid"].to_numpy()
container = toy["Food Container"].to_numpy()
cos_crate_lid = (crate @ lid) / (np.linalg.norm(crate) * np.linalg.norm(lid))
cos_crate_container = (crate @ container) / (
    np.linalg.norm(crate) * np.linalg.norm(container)
)
print(f"\ncosine(Storage Crate, Crate Lid) = {cos_crate_lid:.3f}")
print(f"cosine(Storage Crate, Food Container) = {cos_crate_container:.3f}")

                 Storage Crate  Food Container  Crate Lid  Chopping Board
Sharma Hardware              3               0          5               0
Metro Mart                   4               0          3               0
Coastal Foods                0               6          0               2
Royal Banquets               0               5          0               3
Prime Wholesale              8               0         10               1

cosine(Storage Crate, Crate Lid) = 0.980
cosine(Storage Crate, Food Container) = 0.000


**Reading it.** Storage Crate and Crate Lid are bought together so consistently (every customer who buys one buys the other, in roughly matching quantities) that their cosine similarity is 0.980, nearly the maximum of 1. Storage Crate and Food Container share no customers at all in this toy example, so their similarity is exactly 0: two products can be popular individually and still have nothing in common, if no one buys both.

### The full matrix

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

X = matrix.to_numpy()
items = matrix.columns.tolist()
item_similarity = cosine_similarity(X.T)
np.fill_diagonal(item_similarity, 0)  # a product is not its own recommendation
sim_table = pd.DataFrame(item_similarity, index=items, columns=items)

target = "P01"  # Storage Crate 50L
top_similar = sim_table[target].sort_values(ascending=False).head(4)
print(f"products bought most like {names[target]!r}:")
for product_id, score in top_similar.items():
    print(f"  {names[product_id]:<22} similarity {score:.3f}")

products bought most like 'Storage Crate 50L':
  Crate Lid (50L)        similarity 0.647
  Crate Lid (80L)        similarity 0.573
  Drum Tap Fitting       similarity 0.555
  Stacking Bin Large     similarity 0.548


- `X = matrix.to_numpy()` takes the numbers out of the DataFrame as a plain NumPy array, 4,516 rows by 24 columns; `items` keeps the product IDs in the same column order.
- `cosine_similarity(X.T)`: scikit-learn's `cosine_similarity` compares the *rows* of what it is given, every row with every other. `.T` (transpose, Chapter 35, section 35.3) turns the columns into rows, so it compares products, and returns a 24 × 24 table.
- `np.fill_diagonal(item_similarity, 0)` sets each product's similarity with itself (always 1) to 0, so it can never come top.
- `{names[target]!r}` prints the name with quotes around it (`!r` asks for the `repr`, Chapter 17, section 17.10); `:<22` pads each name to 22 characters so the numbers line up.

**Reading it.** With no product descriptions involved anywhere in this calculation, the top match for Storage Crate 50L is its own lid, the same pair Chapter 38, section 38.9 found (2,800 orders contain both; lift 3.44). The next matches, Crate Lid (80L) and the Drum Tap Fitting, are products bought by similar *kinds* of customers (frequent, larger-volume buyers) rather than obviously related items. That's the signature of collaborative filtering: sometimes it rediscovers an obvious pairing, and sometimes it surfaces a pairing that makes business sense once you see it, but that no one would have listed by looking at the product alone.

### The other way round: user-based filtering

**User-based collaborative filtering** flips the question: find the accounts most similar to this account (cosine similarity on the *rows*) and recommend what they bought. It's the same function without the `.T`:

In [9]:
account_similarity = cosine_similarity(X)
print(account_similarity.shape)

(4516, 4516)


That table has one cell for every pair of accounts: over 20 million numbers, against 576 for the products. Item-based filtering is usually preferred when there are far more customers than products, as here: the product table is small, it changes slowly, and each product's similarities rest on thousands of customers rather than on one account's short history. The rest of this chapter uses item-based filtering.

---

## 42.3 Evaluating recommendations: hit rate, precision, NDCG

### Why accuracy doesn't apply

A recommender doesn't predict one right answer; it produces a **ranked list**, and success means a good item appears *somewhere near the top* of that list. The measures below all look at the top *k* items of the list, and are written with "@k", as in "precision@5". To test a recommender, this chapter hides one product each account really bought and checks where it lands, so each account has exactly one relevant item. Chapter 39, section 39.1 defined precision and recall for a classifier; the ranked-list versions are:

- **Hit rate@k**: the share of customers with at least one relevant item in their top *k*.
- **Recall@k**: relevant items found in the top *k* ÷ relevant items that exist. With one hidden item per customer, recall@k is either 0 or 1, and its average is exactly the hit rate@k.
- **Precision@k**: relevant items found in the top *k* ÷ *k*. With one hidden item, a 5-item list can find at most 1, so precision@5 can never be more than 1 ÷ 5 = 20%.
- **NDCG@k** (normalized discounted cumulative gain): like the hit rate, but a hit near the top counts more than a hit further down. A hit at position *p* (counting from 1) is worth 1 ÷ log₂(*p* + 1): position 1 is worth 1.000, position 2 is worth 0.631, position 3 is worth 0.500, position 5 only 0.387. Adding up those values for every relevant item in the top *k* gives the **DCG**; dividing it by the best possible DCG, the **ideal DCG**, puts it between 0 and 1. With several relevant items, the ideal list puts them all at the top; here there is only one, so the ideal DCG is 1.
- **MRR@k** (mean reciprocal rank): a hit at position *p* scores 1 ÷ *p* (position 2 scores 0.5, position 5 scores 0.2), a miss scores 0, and MRR is the average over customers.

### The metrics by hand

Take one customer whose hidden product comes second in a top-5 list:

In [10]:
relevance = [0, 1, 0, 0, 0]  # the hidden product is at position 2
dcg = sum(
    rel / np.log2(position + 1)
    for position, rel in enumerate(relevance, start=1)
)
ideal = 1 / np.log2(1 + 1)  # best possible: the hidden product at position 1
print(f"hit: {max(relevance)}   precision@5: {sum(relevance) / 5:.0%}")
print(f"DCG: {dcg:.3f}   ideal DCG: {ideal:.3f}   NDCG: {dcg / ideal:.3f}")
print(f"reciprocal rank: {1 / 2:.3f}")
print(f"a hit at position 5 would score NDCG {1 / np.log2(5 + 1):.3f}")

hit: 1   precision@5: 20%
DCG: 0.631   ideal DCG: 1.000   NDCG: 0.631
reciprocal rank: 0.500
a hit at position 5 would score NDCG 0.387


- `enumerate(relevance, start=1)` pairs each item with its position, counting from 1 (Chapter 17, section 17.6; without `start=1` it counts from 0).
- `sum(... for ...)` adds up one value per position: 1 ÷ log₂(position + 1) where the item is relevant, 0 where it isn't.
- `np.log2` is the base-2 logarithm, the log₂ in the formula.

**Reading it.** This one list is a hit (hit rate 1), has precision@5 of 20% (one relevant item in five slots), NDCG 0.631, and reciprocal rank 0.5. The same hit pushed to position 5 would be worth only 0.387, a 61% discount, because position 5 is much less useful to act on than position 1 (a rep with time for one call wants the top suggestion to be right). That's why NDCG is reported alongside the hit rate: it rewards getting the *order* right, not just the presence of a good item somewhere in the list. The two can disagree, and you'll see them disagree below.

**MAP, for lists with several relevant items.** When a customer has several relevant items, **average precision** (Chapter 39, section 39.2) takes the precision at each hit and averages them. Three relevant items at positions 1, 3 and 6 of a top-6 list give precisions 1 ÷ 1, 2 ÷ 3 and 3 ÷ 6, so AP = (1 + 0.667 + 0.5) ÷ 3 = **0.722**. **MAP** (mean average precision) averages AP over customers. With one relevant item at position *p*, AP is just 1 ÷ *p*, so in this chapter MAP and MRR are the same number.

### The evaluation setup: leave-one-out

To test a recommender honestly, hide one thing each customer actually bought, ask the recommender to guess it back from everything else the customer bought, and check where it lands in the top *k*. This is **leave-one-out evaluation**. It is not the same as leave-one-out *cross-validation* (mentioned with `RidgeCV` in Chapter 37, section 37.2): here we hide one item per customer, once, and score the whole ranking.

First, choose the product to hide for each account:

In [11]:
rng = np.random.default_rng(42)
n_owned = (X > 0).sum(axis=1)
eligible = np.where(n_owned >= 2)[0]  # accounts with 2+ products
held_out = np.full(X.shape[0], -1)
for i in eligible:
    owned = np.where(X[i] > 0)[0]
    held_out[i] = rng.choice(owned)
print(f"{len(eligible):,} of {X.shape[0]:,} accounts have 2+ products and can be evaluated")
print("first account's owned product columns:", np.where(X[eligible[0]] > 0)[0])
print("the one hidden:", held_out[eligible[0]])

4,283 of 4,516 accounts have 2+ products and can be evaluated
first account's owned product columns: [ 0  7  8 10 21 22]
the one hidden: 0


- `np.random.default_rng(42)` is a seeded random generator (Chapter 21, section 21.5): the seed 42 makes the "random" choices the same every run, so your numbers match the book's.
- `(X > 0).sum(axis=1)` counts, along each row, how many different products each account bought.
- `np.where(condition)[0]`: `np.where` returns the positions where a condition is True, wrapped in a tuple with one entry per dimension; `[0]` takes the row positions. `eligible` is the rows of accounts with at least 2 products, because an account with one product has nothing left once it is hidden.
- `np.full(X.shape[0], -1)` makes an array with one slot per account, all set to −1 ("nothing hidden").
- `rng.choice(owned)` picks one of the account's product columns at random.
- The last two lines show the first eligible account: it owns six products, and the one hidden is column 0. Columns count from 0, so column 0 is P01, the Storage Crate 50L.

Then remove the hidden products from a copy of the matrix. Every method in this chapter learns from `X_train` only:

In [12]:
X_train = X.copy()
X_train[eligible, held_out[eligible]] = 0
print(f"{X_train.sum():,.0f} units remain of {X.sum():,.0f} original")

1,489,129 units remain of 1,617,820 original


- `X.copy()` makes a separate array, so `X` keeps the full data.
- `X_train[eligible, held_out[eligible]] = 0` sets one cell per eligible account to 0: row `eligible[0]`, column `held_out[eligible[0]]`, and so on. Two arrays of positions pick pairs of cells, one per account.

### One account, by hand

Before scoring 4,283 accounts, score one. The popularity recommender gives every product a score equal to the number of accounts that bought it, the same for everyone:

In [13]:
pop_counts = (X_train > 0).sum(axis=0).astype(float)

i = eligible[0]
scores = pop_counts.copy()
scores[X_train[i] > 0] = -np.inf  # products this account owns go to the bottom
top_5 = np.argsort(-scores)[:5]
print("top 5:", [names[items[j]] for j in top_5])
print("hidden:", names[items[held_out[i]]])
if held_out[i] in top_5:
    print("hit at position", int(np.where(top_5 == held_out[i])[0][0]) + 1)
else:
    print("miss: the hidden product is not in the top 5")

top 5: ['Crate Lid (80L)', 'Crate Lid (50L)', 'Stacking Bin Large', 'Food Container 2L', 'Serving Tray']
hidden: Storage Crate 50L
miss: the hidden product is not in the top 5


- `pop_counts` counts, for each product, the accounts that bought it in `X_train`. `.astype(float)` makes it a float array, because the next lines put `-np.inf` into a copy of it, and an integer array can't hold infinity.
- `scores = pop_counts.copy()` copies it, so this account's changes don't touch the shared counts.
- `scores[X_train[i] > 0] = -np.inf` gives every product the account already owns a score of minus infinity, lower than any real score, so it sorts to the very bottom and is never recommended: re-recommending something a customer already buys isn't useful.
- `np.argsort(-scores)` gives the column positions that would sort `scores` from smallest to largest; sorting the negatives puts the largest score first. `[:5]` keeps the top five.
- `held_out[i] in top_5` asks whether the hidden product made the list. If it did, `np.where(top_5 == held_out[i])[0][0]` finds where, counted from 0, and `+ 1` turns that into a position counted from 1.

For this account, popularity is a miss: it suggests the lids, bins and food containers that most accounts buy, and the account's hidden Storage Crate 50L isn't among them.

### Every account at once

The function below repeats those steps for every eligible account and averages the three measures. Its first argument, `score_fn`, is itself a function: *any* function that takes an account's row number and returns 24 scores, one per product. That lets one `evaluate` test every recommender in this chapter.

In [14]:
def evaluate(score_fn, k=5):
    hits = 0
    ndcg_total = 0.0
    rr_total = 0.0
    for i in eligible:
        scores = score_fn(i).copy()
        scores[X_train[i] > 0] = -np.inf  # never recommend something already bought
        top_k = np.argsort(-scores)[:k]
        if held_out[i] in top_k:
            position = int(np.where(top_k == held_out[i])[0][0]) + 1
            hits += 1
            ndcg_total += 1 / np.log2(position + 1)
            rr_total += 1 / position
    n = len(eligible)
    return hits / n, ndcg_total / n, rr_total / n


hit_pop, ndcg_pop, mrr_pop = evaluate(lambda i: pop_counts)
print(
    f"popularity baseline:  hit rate@5 {hit_pop:.1%}   "
    f"NDCG@5 {ndcg_pop:.3f}   MRR@5 {mrr_pop:.3f}"
)
print(f"its precision@5 is the hit rate ÷ 5: {hit_pop / 5:.1%}")

popularity baseline:  hit rate@5 62.7%   NDCG@5 0.434   MRR@5 0.372
its precision@5 is the hit rate ÷ 5: 12.5%


- `k=5` is a default value (Chapter 17, section 17.8): call `evaluate(f)` for the top 5, or `evaluate(f, k=10)` for the top 10.
- `hits`, `ndcg_total` and `rr_total` add up each account's hit (1 or 0), its NDCG (1 ÷ log₂(position + 1)) and its reciprocal rank (1 ÷ position). A miss adds 0 to all three.
- The `return` line divides each total by the number of accounts, giving the hit rate@k, NDCG@k and MRR@k.
- `lambda i: pop_counts` is a function without a name (Chapter 18, section 18.6) that takes an account's row number `i` and returns the popularity counts. It ignores `i`, because popularity is the same for everyone.

**Reading it.** The popularity baseline puts the hidden product in the top 5 for 62.7% of accounts. That's high because there are only 24 products and each account already owns several of them, so five suggestions cover a large share of what's left. As a precision@5 it's 12.5%, well under the 20% ceiling. From here on, the chapter reports hit rate@5, NDCG@5 and MRR@5.

### Item-based collaborative filtering, tested

To score products for one account, each product the account owns "votes" for the products similar to it, weighted by how much of it the account bought:

In [15]:
item_sim_train = cosine_similarity(X_train.T)
np.fill_diagonal(item_sim_train, 0)


def item_cf_score(i):
    return X_train[i] @ item_sim_train


hit_cf, ndcg_cf, mrr_cf = evaluate(item_cf_score)
print(
    f"item-based collaborative filtering: hit rate@5 {hit_cf:.1%}   "
    f"NDCG@5 {ndcg_cf:.3f}   MRR@5 {mrr_cf:.3f}"
)

item-based collaborative filtering: hit rate@5 65.0%   NDCG@5 0.495   MRR@5 0.443


- `item_sim_train` is the 24 × 24 product similarity table again, built from `X_train` this time, so the hidden products play no part.
- `X_train[i] @ item_sim_train` is a vector times a matrix (Chapter 35, section 35.3). For each product, it adds up *quantity of each owned product × that product's similarity to it*. A product similar to many things the account buys a lot of gets a high score.

**Reading it.** Item-based collaborative filtering beats popularity (65.0% against 62.7% hit rate@5; NDCG 0.495 against 0.434): using *who bought what together* improves on suggesting the same popular items to everyone.

### Quantities or yes/no?

The similarities above use raw quantities, so an account that buys 400 crate lids counts 400 times as much as one that buys 1, and a few very large wholesale buyers can dominate. Two common fixes: use only *whether* each account bought each product (`X_train > 0`), or shrink big quantities with the log transform, `np.log1p` (Chapter 36, section 36.6). Both, tested the same way:

In [16]:
variant_results = {}
for label, weights in [
    ("bought yes/no", (X_train > 0).astype(float)),
    ("log(1 + quantity)", np.log1p(X_train)),
]:
    sim = cosine_similarity(weights.T)
    np.fill_diagonal(sim, 0)
    variant_results[label] = evaluate(lambda i: weights[i] @ sim)
    hit, ndcg, mrr = variant_results[label]
    print(f"item CF on {label:<18} hit rate@5 {hit:.1%}   NDCG@5 {ndcg:.3f}   MRR@5 {mrr:.3f}")

item CF on bought yes/no      hit rate@5 68.7%   NDCG@5 0.504   MRR@5 0.443


item CF on log(1 + quantity)  hit rate@5 68.3%   NDCG@5 0.510   MRR@5 0.453


- `(X_train > 0).astype(float)` turns every quantity into 1.0 (bought) or 0.0 (not bought).
- `np.log1p(X_train)` is ln(1 + quantity): 0 stays 0, 9 becomes 2.3, 399 becomes 6.0.
- The loop builds a similarity table from each version, evaluates it, and keeps the result in `variant_results` for section 42.7's scoreboard. The lambda uses `weights` and `sim` from the current pass of the loop; `evaluate` runs it straight away, before the loop moves on (section 42.4 shows why that matters).

**Reading it.** Both versions beat raw quantities: 68.7% and 68.3% hit rate@5 against 65.0%, and the log version has the best NDCG of the three (0.510). The biggest buyers were drowning out everyone else. Section 42.4 builds on that idea; for simplicity, the rest of the chapter keeps the raw-quantity `item_cf_score` as its item-based method, and the final scoreboard in section 42.7 lists the yes/no version too.

---

## 42.4 Matrix factorization and implicit feedback

### The idea

**Matrix factorization** approximates the account-by-product matrix with two smaller matrices, an account-by-factor matrix and a factor-by-product matrix, such that multiplying them back together approximately reconstructs the original. The **factors** are learned, not chosen: they might end up meaning something like "buys a lot of industrial gear" or "buys kitchenware in small quantities", but nobody tells the model what to call them, exactly as PCA's components in Chapter 35, section 35.10 had no names until you looked at their loadings. Because they are hidden in the data rather than measured, they are called **latent factors**. Chapter 41, section 41.6's NMF drew the same picture for documents and topics.

### Factorization by hand

Take section 42.2's toy table. Two factors look likely: a "storage" factor (crates and lids) and a "food" factor (containers and boards). Write each account's score on the two factors in a 5 × 2 table **A**, and each product's weight on the two factors in a 2 × 4 table **B**:

In [17]:
A = pd.DataFrame(
    [[6, 0], [5, 0], [0, 6.3], [0, 5.8], [13, 0]],
    index=toy.index,
    columns=["storage", "food"],
)
B = pd.DataFrame(
    [[0.6, 0, 0.8, 0.1], [0, 0.9, 0, 0.4]],
    index=["storage", "food"],
    columns=toy.columns,
)
print((A @ B).round(1))

                 Storage Crate  Food Container  Crate Lid  Chopping Board
Sharma Hardware            3.6             0.0        4.8             0.6
Metro Mart                 3.0             0.0        4.0             0.5
Coastal Foods              0.0             5.7        0.0             2.5
Royal Banquets             0.0             5.2        0.0             2.3
Prime Wholesale            7.8             0.0       10.4             1.3


- `A` holds each account's two factor scores: Sharma Hardware is 6 on storage and 0 on food; Coastal Foods is 0 and 6.3.
- `B` holds each product's weight on each factor: Crate Lid has 0.8 on storage and 0 on food; Chopping Board has a little of both (0.1 and 0.4).
- `A @ B` multiplies them (Chapter 35, section 35.3): a 5 × 2 table times a 2 × 4 table gives a 5 × 4 table, the same shape as `toy`.

Check one cell by hand. Sharma Hardware × Storage Crate = 6 × 0.6 + 0 × 0 = 3.6, against 3 in the real table: close. The 20 numbers of the toy table have been summed up by 18 (10 in A, 8 in B), and the rebuilt table is near the original everywhere. Now look at Sharma Hardware × Chopping Board: 6 × 0.1 + 0 × 0.4 = 0.6. Sharma has never bought a chopping board (the real cell is 0), but the reconstruction gives it a small positive score, because Prime Wholesale, another storage buyer, bought one. **That non-zero score in an empty cell is the recommendation.** Matrix factorization recommends by rebuilding the matrix and reading the scores it puts in the cells that were empty.

Nobody picks A and B by hand in practice. scikit-learn's **`TruncatedSVD`** finds them:

In [18]:
from sklearn.decomposition import TruncatedSVD

toy_svd = TruncatedSVD(n_components=2, random_state=42).fit(toy)
print(toy_svd.transform(toy).round(2))
print(toy_svd.components_.round(2))

[[ 5.76 -0.2 ]
 [ 4.84 -0.17]
 [ 0.19  6.29]
 [ 0.24  5.79]
 [12.84 -0.04]]
[[ 0.63  0.01  0.77  0.06]
 [-0.02  0.91 -0.03  0.41]]


- `TruncatedSVD(n_components=2)` asks for two factors; `random_state=42` fixes the random start of its search, so the result repeats. `.fit(toy)` learns them.
- `.transform(toy)` returns table A, each account's two factor scores; `.components_` is table B, each factor's weight on each product (a learned attribute, with the trailing `_` of Chapter 36, section 36.4).

The library's tables have the same pattern as the hand-made ones, with different scaling: the first factor weighs crates and lids, the second containers and boards, and the small negative numbers are near-zero noise. **TruncatedSVD is PCA without first subtracting the column means** (Chapter 35, section 35.10): because it doesn't center, a 0 stays a 0, which suits a matrix that is mostly zeros.

### On the real matrix

First, fit once with four factors and look at the shapes:

In [19]:
svd = TruncatedSVD(n_components=4, random_state=42).fit(X_train)
account_factors = svd.transform(X_train)
reconstructed = account_factors @ svd.components_
print("account factors:", account_factors.shape)
print("product factors:", svd.components_.shape)
print("reconstructed:  ", reconstructed.shape)
print(f"share of the variance kept: {svd.explained_variance_ratio_.sum():.1%}")

account factors: (4516, 4)
product factors: (4, 24)
reconstructed:   (4516, 24)
share of the variance kept: 63.1%


- `account_factors` is table A for the real data: 4,516 accounts × 4 factors. `svd.components_` is table B: 4 factors × 24 products.
- `reconstructed` multiplies them back into a 4,516 × 24 table of scores, one for every account and product, including the empty cells.
- `explained_variance_ratio_` gives each factor's share of the variation in the data, as for PCA; `.sum()` adds them up.

Next, test several factor counts with `evaluate`. Each `score_fn` must remember *this* loop's `reconstructed`, and a function defined inside a loop doesn't do that on its own.

### Freezing a loop's value

Python looks up a name inside a function when the function *runs*, not when it's made. Here are three small functions made in a loop, then called afterwards:

In [20]:
late = [lambda: k for k in range(3)]
frozen = [lambda k=k: k for k in range(3)]
print([f() for f in late], [f() for f in frozen])

[2, 2, 2] [0, 1, 2]


All three `late` functions look up `k` after the loop has finished, when `k` is 2. `lambda k=k: k` gives each function a default value (Chapter 17, section 17.8), and a default value is stored the moment the function is made, so each one keeps its own `k`. The loop below uses the same trick: `r=reconstructed` stores this pass's matrix inside the function. `evaluate` happens to run each function straight away, so here it would work either way; the default argument keeps it safe if you store the functions and call them later.

In [21]:
svd_results = {}
for n_factors in [4, 8, 12]:
    svd = TruncatedSVD(n_components=n_factors, random_state=42).fit(X_train)
    reconstructed = svd.transform(X_train) @ svd.components_
    svd_results[n_factors] = evaluate(lambda i, r=reconstructed: r[i])
    hit, ndcg, mrr = svd_results[n_factors]
    print(
        f"{n_factors:>2} factors: hit rate@5 {hit:.1%}   NDCG@5 {ndcg:.3f}   "
        f"MRR@5 {mrr:.3f}   variance kept {svd.explained_variance_ratio_.sum():.1%}"
    )

 4 factors: hit rate@5 64.7%   NDCG@5 0.486   MRR@5 0.434   variance kept 63.1%
 8 factors: hit rate@5 56.4%   NDCG@5 0.400   MRR@5 0.346   variance kept 74.7%


12 factors: hit rate@5 46.4%   NDCG@5 0.317   MRR@5 0.268   variance kept 83.4%


- `svd_results = {}` is an empty dictionary; each pass stores its three measures under its factor count, for the scoreboard in section 42.7.
- `lambda i, r=reconstructed: r[i]` returns row `i` of this pass's reconstructed table: account `i`'s score for every product.
- `{n_factors:>2}` right-aligns the number in two characters, so the lines line up.

**Reading it, plainly.** More factors capture more of the matrix's variance (up to 83% at 12 factors), and the recommendations get **worse**, not better: the hit rate@5 falls from 64.7% to 46.4% as factors rise from 4 to 12. With only 24 products, a handful of factors already explains most of the real structure; extra factors fit noise. This is Chapter 37's bias-variance trade-off again (section 37.10), in a new setting: more capacity is not automatically better, and the honest way to know is to test it.

### Implicit feedback

A five-star rating is **explicit feedback**: the customer tells you directly how much they liked something. Riverstone's data has no ratings, only purchase quantities: **implicit feedback**, where preference has to be inferred from behavior. A customer buying 40 units of something is a much stronger signal than buying 1, but it isn't a rating: someone who buys 40 crate lids isn't "40 times more satisfied" than someone who buys 1, they just run a bigger business. Section 42.3 saw the same thing: item-based filtering did better once big quantities were shrunk.

The standard approach, Hu, Koren and Volinsky's method, implemented in the `implicit` library, treats each cell as two things: a yes/no **preference** (did they buy it at all?) and a **confidence** in that yes or no, which grows with the quantity. Their paper uses confidence = 1 + α × quantity, with α a number you choose; here we pass log(1 + quantity) as the confidence directly, a simple variant that dampens large quantities exactly as section 42.3 did.

The method is **alternating least squares (ALS)**. It looks for the same two tables A and B, and "alternating" describes how: fix the product table B and solve for the best account table A by least squares (Chapter 37, section 37.1, one small regression per account); then fix A and solve for B; repeat. Each step can only improve the fit, and a few rounds are enough.

`implicit` wants its input as a **sparse matrix** (Chapter 36, section 36.4), a format that stores only the non-zero cells. SciPy's `csr_matrix` makes one from a NumPy array:

In [22]:
from scipy.sparse import csr_matrix
from threadpoolctl import threadpool_limits

threadpool_limits(1, "blas")  # the setting implicit asks for (see below)
confidence = csr_matrix(np.log1p(X_train))
print(repr(confidence))

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 37525 stored elements and shape (4516, 24)>


- `csr_matrix(...)` keeps each non-zero value with its row and column; rows are accounts, columns are products, the format `implicit` expects. `repr` shows its shape and how many cells it actually stores: 37,525 of the 108,384 (4,516 × 24).
- `threadpool_limits(1, "blas")`: NumPy does matrix arithmetic with a library called BLAS, which can split work across several processor threads. `implicit` runs its own threads, and warns that the two together slow each other down, so it asks for BLAS to use one thread. `threadpoolctl` came with scikit-learn, so there's nothing to install.

Now fit ALS with 8 and 16 factors:

In [ ]:
als_results = {}
for n_factors in [8, 16]:
    model = implicit.als.AlternatingLeastSquares(
        factors=n_factors, regularization=0.1, iterations=20, random_state=42
    )
    model.fit(confidence, show_progress=False)

    def als_score(i, m=model):
        return m.user_factors[i] @ m.item_factors.T

    als_results[n_factors] = evaluate(als_score)
    hit, ndcg, mrr = als_results[n_factors]
    print(
        f"ALS, {n_factors:>2} factors: hit rate@5 {hit:.1%}   "
        f"NDCG@5 {ndcg:.3f}   MRR@5 {mrr:.3f}"
    )

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
ALS,  8 factors: hit rate@5 67.9%   NDCG@5 0.488   MRR@5 0.425
ALS, 16 factors: hit rate@5 46.0%   NDCG@5 0.344   MRR@5 0.306
```

- `factors` is the number of columns in A and rows in B, as in the toy example.
- `regularization` is a penalty on large factor values, the same idea as Chapter 37, section 37.2's ridge penalty: it keeps the factors small and stable.
- `iterations` is how many times ALS alternates between solving A and solving B.
- `random_state=42` fixes the random starting values, so the result repeats.
- `model.fit(confidence, show_progress=False)` learns the two tables; `show_progress=False` hides the progress bar it would otherwise draw.
- `m.user_factors` is table A (one row per account; `implicit` calls them users) and `m.item_factors` is table B turned on its side (one row per product). `m.user_factors[i] @ m.item_factors.T` multiplies account `i`'s factor row by every product's factor row: one predicted score per product. `m=model` freezes this pass's model, as shown above.

What happens if you change them? `factors` is the setting that matters most here: from 8 to 16, the hit rate falls by more than 20 points, as the output shows. Exercise 7 changes `regularization` and finds almost no difference.

**Reading it.** ALS with 8 factors (67.9% hit rate@5) beats raw-quantity item-based filtering (65.0%) and the best SVD (64.7%), but not item-based filtering on yes/no data (68.7%), which also has the higher NDCG (0.504 against 0.488). And ALS's NDCG is below even raw item-based filtering's 0.495, although its hit rate is higher: the two metrics can disagree, which is why you report both. With 16 factors, ALS collapses further than the SVD did (46.0%), the same overfitting pattern, more pronounced. **The lesson generalizes beyond this dataset:** on a small or narrow catalog, keep factor counts low and always check with a held-out test, rather than trusting a bigger, fancier-sounding model by default.

---

## 42.5 Content-based filtering

### Reusing Chapter 41

Every catalog item has a short text description (section 42.1). Turning those into TF-IDF vectors and comparing them with cosine similarity, precisely Chapter 41, section 41.3's pipeline, gives a recommender that needs no purchase history at all. First, the similarities:

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

description_by_item = descriptions.loc[items]
tfidf = TfidfVectorizer(stop_words="english")
description_vectors = tfidf.fit_transform(description_by_item)
content_similarity = cosine_similarity(description_vectors)
content_table = pd.DataFrame(content_similarity, index=items, columns=items)
top_content = content_table["P01"].drop("P01").sort_values(ascending=False).head(4)
print("most similar products to Storage Crate 50L by description text alone:")
for product_id, score in top_content.items():
    print(f"  {names[product_id]:<22} similarity {score:.3f}")

most similar products to Storage Crate 50L by description text alone:
  Storage Crate 80L      similarity 0.539
  Crate Lid (50L)        similarity 0.296
  Food Container 5L      similarity 0.249
  Food Container 2L      similarity 0.249


- `descriptions.loc[items]` puts the descriptions in the same order as the matrix columns, so product *j*'s text lines up with column *j*.
- `TfidfVectorizer(stop_words="english")` drops common English words and weighs the rest by TF-IDF, as in Chapter 41. Its idf is learned from just these 24 descriptions, so a word that appears in many of them, such as "plastic", counts for little.
- `cosine_similarity(description_vectors)` compares every description with every other: a 24 × 24 table. Its diagonal (a product with itself) is 1; that's harmless here, because `evaluate` removes owned products anyway.
- `.drop("P01")` leaves the crate itself out of its own list.

Then the recommender, in the same shape as item-based filtering, with text similarity in place of purchase similarity:

In [25]:
def content_score(i):
    return X_train[i] @ content_similarity


hit_content, ndcg_content, mrr_content = evaluate(content_score)
print(
    f"content-based filtering: hit rate@5 {hit_content:.1%}   "
    f"NDCG@5 {ndcg_content:.3f}   MRR@5 {mrr_content:.3f}"
)

content-based filtering: hit rate@5 53.0%   NDCG@5 0.357   MRR@5 0.300


**Reading it.** By description alone, Storage Crate 50L's nearest neighbor is correctly its 80-litre sibling (they share almost every word except the size), but its matching lid drops to third, behind two food containers that happen to share generic words like "litre", "storage", "food safe" and "plastic". Content-based similarity captures *what a product is*, not *what customers actually do with it together*: it doesn't know that a crate and its lid are bought as a pair, only that their descriptions read somewhat alike. Its hit rate@5, 53.0%, is the weakest of the methods so far apart from the 12-factor SVD and the 16-factor ALS, behind even the popularity baseline.

> **Watch out: content-based filtering is limited by how good the descriptions are.** These 24 descriptions were written once, by hand, with real product differences. Real catalogs often have sparse, inconsistent, or auto-generated descriptions, and a content recommender is only ever as good as the text it's given (Chapter 41's cleaning lessons apply directly here).

### Hybrid: combining both

To blend two recommenders, add their scores. But the two are measured in different units: quantity × purchase similarity against quantity × text similarity. Compare their largest scores for three accounts:

In [26]:
for i in eligible[:3]:
    print(
        f"account row {i}: item CF up to {item_cf_score(i).max():.0f}, "
        f"content up to {content_score(i).max():.0f}"
    )

account row 0: item CF up to 71, content up to 62
account row 1: item CF up to 105, content up to 84
account row 2: item CF up to 6, content up to 10


Which one is bigger changes from account to account, so in a raw sum whichever happened to be larger would decide the blend. So each score list is divided by its own largest absolute value first, which puts both on a 0-to-1 scale, and then the two are mixed:

In [27]:
def normalize(scores):
    spread = np.abs(scores).max()
    return scores / spread if spread > 0 else scores


def hybrid_score(i, weight=0.5):
    collaborative = normalize(item_cf_score(i))
    content = normalize(content_score(i))
    return weight * collaborative + (1 - weight) * content


hit_hybrid, ndcg_hybrid, mrr_hybrid = evaluate(hybrid_score)
print(
    f"hybrid (50/50 collaborative + content): hit rate@5 {hit_hybrid:.1%}   "
    f"NDCG@5 {ndcg_hybrid:.3f}   MRR@5 {mrr_hybrid:.3f}"
)

hybrid (50/50 collaborative + content): hit rate@5 63.7%   NDCG@5 0.448   MRR@5 0.386


- `normalize` divides by the largest absolute value, `np.abs(scores).max()`, so the biggest score becomes 1. The `if spread > 0 else scores` guards against dividing by zero for an account whose scores are all 0.
- `weight=0.5` is a default value (Chapter 17, section 17.8): `hybrid_score(i)` mixes 50% collaborative with 50% content, and `hybrid_score(i, weight=0.8)` would use 80% and 20%.

**Reading it.** A straight 50/50 blend of collaborative and content scores (63.7% hit rate@5) lands between the two on their own, not above the better one. Blending isn't automatically an improvement; it's a way to **trade off** two different kinds of failure, worth doing when you need the content signal for cold-start cases (next section), but not worth doing purely to chase a higher headline number when one method is already clearly ahead on its own.

---

## 42.6 The cold-start problem

Every collaborative method in this chapter has the same blind spot: it needs purchase history to work at all. Two situations break it completely.

**A brand-new product**, never purchased by anyone, has an all-zero column in the interaction matrix: every collaborative similarity to it is zero or undefined. Content-based filtering has no such problem, because a description exists on day one. First, add the new product's description to the other 24:

In [28]:
new_product = pd.DataFrame(
    [
        {
            "product_id": "P99",
            "product_name": "Recycled Plastic Crate 50L",
            "description": (
                "Eco-friendly recycled plastic storage crate, 50 litre, "
                "food safe, ventilated sides."
            ),
        }
    ]
)
extended_descriptions = pd.concat(
    [description_by_item.reset_index(), new_product[["product_id", "description"]]],
    ignore_index=True,
)
print(len(extended_descriptions), "descriptions; the last two:")
print(extended_descriptions.tail(2).to_string())

25 descriptions; the last two:
   product_id                                                                          description
23        P24         Stackable plastic shoe rack, 3 tier, for home or hospitality entrance areas.
24        P99  Eco-friendly recycled plastic storage crate, 50 litre, food safe, ventilated sides.


- `new_product` is a one-row table built from a list holding one dictionary.
- `description_by_item.reset_index()` turns section 42.5's Series into a two-column table, `product_id` and `description`, so it matches the new row's columns.
- `pd.concat([...], ignore_index=True)` stacks the two tables (Chapter 18, section 18.7) and numbers the rows 0 to 24 afresh.

Then vectorize all 25 and read the new product's row:

In [29]:
extended_vectors = TfidfVectorizer(stop_words="english").fit_transform(
    extended_descriptions["description"]
)
extended_similarity = cosine_similarity(extended_vectors)
new_index = len(extended_descriptions) - 1
nearest_to_new = (
    pd.Series(extended_similarity[new_index][:-1], index=items)
    .sort_values(ascending=False)
    .head(3)
)
print("a brand-new product, never purchased, matched by description alone:")
for product_id, score in nearest_to_new.items():
    print(f"  {names[product_id]:<22} similarity {score:.3f}")

a brand-new product, never purchased, matched by description alone:
  Storage Crate 50L      similarity 0.581
  Storage Crate 80L      similarity 0.449
  Food Container 2L      similarity 0.232


- A new `TfidfVectorizer` is fitted on all 25 descriptions, so the new product's words, such as "recycled", are in the vocabulary. That also changes every word's idf slightly, so these scores are not directly comparable with section 42.5's. In production you'd refit it whenever products are added, for example nightly.
- `new_index = len(extended_descriptions) - 1` is the new product's row, the last one (24).
- `extended_similarity[new_index][:-1]` takes that row and drops its last entry, the new product's similarity with itself.

**Reading it.** A newly launched recycled-plastic crate, with zero sales, is correctly matched to the existing Storage Crate 50L and 80L by its description alone. No collaborative method in this chapter could recommend it to anyone until it had already been bought a meaningful number of times: a genuine chicken-and-egg problem. This is the single strongest argument for keeping a content-based component in any production recommender, even when it scores lower on its own: it's the only one of these methods that works on day one of a launch.

**A brand-new customer** has the mirror problem: an all-zero row, so collaborative filtering has nothing to go on. The standard fixes are the same ones used elsewhere in this book: fall back to the **popularity baseline** (section 42.1) until enough history accumulates, ask a few onboarding questions and match against **segment-level** behavior (the next section), or use whatever *is* known (company size, industry, city) as a substitute for purchase history, the same idea as Chapter 36's features standing in for a target that hasn't happened yet.

---

## 42.7 B2B cross-selling and ranking by segment

### Turning a co-purchase pattern into an action

Chapter 38 found market basket rules; here's the same idea, framed as a specific, assignable cross-sell action. Which products share an order with the Industrial Crate 200L?

In [30]:
order_baskets = lines.groupby("order_id")["product_id"].apply(set)
target_item = "P13"  # Industrial Crate 200L
contains_target = order_baskets.apply(lambda s: target_item in s)
partner_counts = pd.Series(
    [
        p
        for basket in order_baskets[contains_target]
        for p in sorted(basket)
        if p != target_item
    ]
).value_counts()
partner_share = (partner_counts / contains_target.sum()).rename(index=names)
print(
    f"of {contains_target.sum():,} orders containing {names[target_item]!r}, "
    f"the other products they also contain:"
)
print((partner_share.head(5) * 100).round(1).astype(str).add("%").to_string())

of 3,397 orders containing 'Industrial Crate 200L', the other products they also contain:
Dolly Wheels (set)    47.5%
Drum Tap Fitting      23.7%
Crate Lid (80L)       17.5%
Crate Lid (50L)       16.8%
Drum 60L              15.6%


- `groupby("order_id")["product_id"].apply(set)` turns each order into a set of product IDs, its basket (as in Chapter 38, section 38.9).
- `contains_target` is True for each basket that holds the crate.
- The list comprehension (Chapter 17, section 17.6) walks through those baskets and collects every *other* product in them; `value_counts()` counts each one, and dividing by the number of crate orders gives a share. `sorted(basket)` walks each basket in ID order: a set has no fixed order, and sorting makes products with equal counts always print in the same order.

**Reading it.** Of the 3,397 orders containing an Industrial Crate 200L, nearly half (47.5%) also include a set of Dolly Wheels, the castors that fit under crates and pallet boxes: a natural add-on. About a quarter include the Drum Tap Fitting, which Chapter 38's rules tied to the Drum 60L, and about one in six includes each crate lid. An account manager working wholesale accounts has a concrete, short list to check before a call: has this crate buyer also picked up the wheels most crate buyers add?

### Ranking differs by who's asking

In [31]:
for segment in ["Retail", "Hospitality", "Wholesale"]:
    segment_accounts = accounts.loc[accounts["segment"] == segment, "account_id"]
    segment_rows = matrix.index.isin(segment_accounts)
    segment_top = bought[segment_rows].mean().sort_values(ascending=False).head(3)
    print(
        f"{segment:<12} top products: "
        + ", ".join(f"{names[p]} ({v:.0%})" for p, v in segment_top.items())
    )

Retail       top products: Airtight Seal Pack (56%), Crate Lid (80L) (54%), Crate Lid (50L) (53%)
Hospitality  top products: Airtight Seal Pack (75%), Food Container 2L (59%), Food Container 5L (59%)
Wholesale    top products: Drum Tap Fitting (90%), Dolly Wheels (set) (89%), Crate Lid (50L) (85%)


- `matrix.index.isin(segment_accounts)` is True for the rows whose account is in this segment.
- `bought[segment_rows].mean()` is each product's share of those accounts that bought it; the top three follow.

**Reading it.** The three segments want quite different top recommendations: Wholesale accounts are led by industrial accessories (drum tap fittings and dolly wheels, each bought by about nine in ten), Hospitality by food storage, Retail by seal packs and crate lids. A single popularity list, ignoring segment, never shows a wholesale account the drum taps and dolly wheels that almost every wholesale account buys, because fewer than 30% of *all* accounts buy them. The cheapest, most robust improvement over plain popularity is often not a fancier algorithm; it's popularity **computed separately for each meaningful group**, exactly as Chapter 38's cluster profiles (section 38.3) suggested treating different customer groups differently.

### Segment-level popularity, tested

Does that simple idea actually rank better? Build popularity counts separately for each segment, from `X_train` so the hidden products stay hidden, and give each account its own segment's list:

In [32]:
segment_of = accounts.set_index("account_id")["segment"]
account_segments = segment_of[matrix.index].to_numpy()
segment_pop_counts = {
    segment: (X_train[account_segments == segment] > 0).sum(axis=0).astype(float)
    for segment in ["Retail", "Hospitality", "Wholesale"]
}


def segment_popularity_score(i):
    return segment_pop_counts[account_segments[i]]


hit_seg, ndcg_seg, mrr_seg = evaluate(segment_popularity_score)
print(
    f"segment-level popularity: hit rate@5 {hit_seg:.1%}   "
    f"NDCG@5 {ndcg_seg:.3f}   MRR@5 {mrr_seg:.3f}"
)

segment-level popularity: hit rate@5 71.5%   NDCG@5 0.533   MRR@5 0.473


- `segment_of[matrix.index]` looks up each matrix row's segment, in row order; `account_segments` is that as an array, so `account_segments[i]` is row `i`'s segment.
- `X_train[account_segments == segment]` keeps the rows of one segment (a True/False array selects rows, Chapter 18, section 18.4), and `(... > 0).sum(axis=0)` counts, per product, the accounts in that segment that bought it.
- The dictionary comprehension (Chapter 17, section 17.7) builds one count array per segment.

### The scoreboard

Every method in this chapter, tested the same way, goes into one table. The yes/no, SVD and ALS rows come from the dictionaries filled in sections 42.3 and 42.4.

In [ ]:
scoreboard = pd.DataFrame(
    {
        "popularity": (hit_pop, ndcg_pop, mrr_pop),
        "segment popularity": (hit_seg, ndcg_seg, mrr_seg),
        "item-based CF": (hit_cf, ndcg_cf, mrr_cf),
        "item CF, yes/no": variant_results["bought yes/no"],
        "SVD, 4 factors": svd_results[4],
        "SVD, 12 factors": svd_results[12],
        "ALS, 8 factors": als_results[8],
        "ALS, 16 factors": als_results[16],
        "content-based": (hit_content, ndcg_content, mrr_content),
        "hybrid 50/50": (hit_hybrid, ndcg_hybrid, mrr_hybrid),
    },
    index=["hit rate@5", "NDCG@5", "MRR@5"],
).T.sort_values("hit rate@5", ascending=False)
print(scoreboard.round(3).to_string())

> **Not run here.** This cell needs something this machine does not have: a database, an API key, a running service, or command-line arguments. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
                    hit rate@5  NDCG@5  MRR@5
segment popularity       0.715   0.533  0.473
item CF, yes/no          0.687   0.504  0.443
ALS, 8 factors           0.679   0.488  0.425
item-based CF            0.650   0.495  0.443
SVD, 4 factors           0.647   0.486  0.434
hybrid 50/50             0.637   0.448  0.386
popularity               0.627   0.434  0.372
content-based            0.530   0.357  0.300
SVD, 12 factors          0.464   0.317  0.268
ALS, 16 factors          0.460   0.344  0.306
```

- Each dictionary entry is one method's three measures; `pd.DataFrame(..., index=[...])` makes them columns, with the three measure names as rows.
- `.T` turns the table on its side, one row per method, and `sort_values("hit rate@5", ascending=False)` puts the best hit rate first.
- The yes/no item-based method comes from section 42.3's `variant_results`, and the SVD and ALS rows from the `svd_results` and `als_results` dictionaries of section 42.4.

**Reading it.** Segment-level popularity tops the board on all three measures (71.5% hit rate@5, NDCG 0.533, MRR 0.473), ahead of every model, with no training at all. Next come item-based filtering on yes/no data and ALS with 8 factors, then raw item-based filtering and the 4-factor SVD, all within four points of each other; then the hybrid, the plain popularity baseline, content-based filtering, and the over-sized factor models at the bottom. The order also depends on the measure: ALS is third on hit rate, but on NDCG it falls behind raw item-based filtering, and on MRR behind the 4-factor SVD too. On a 24-product catalog where the segment explains most of why accounts buy differently, knowing an account's segment is worth more than any pattern the models find in its purchases. That won't hold for every catalog, which is exactly why you test it.

---

## Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Using accuracy or a plain train/test split | Metric doesn't fit a ranked-list problem | Use hit rate@k and NDCG@k with leave-one-out evaluation |
| Calling a hit rate "precision" | "Precision@5 of 65%" when one hidden item caps precision@5 at 20% | Name the measure you computed: hit rate@k, recall@k or precision@k |
| Recommending items a customer already owns | Rep annoyed: "they already buy that" | Exclude owned items from the candidate list before ranking |
| Assuming more matrix-factorization factors is always better | Hit rate falls as factor count rises | Test several factor counts on held-out data; small catalogs need few factors |
| Content-based filtering with poor or missing descriptions | Recommendations feel random | Improve the underlying text (Chapter 41); check similarity scores make sense on known pairs |
| No plan for new products or new customers | New launches get no recommendations for months | Combine collaborative filtering with a content-based or popularity fallback |
| One global popularity list for every customer | Segment-mismatched suggestions | Compute popularity, or any baseline, per meaningful group |
| Treating quantity as a five-star rating | Big-volume customers look "more satisfied" than they are | Use yes/no or log quantities, or implicit-feedback methods (confidence, not rating) |
| Blending models without checking it helps | Hybrid score is worse than the best single method | Evaluate the blend against each component on held-out data before adopting it |
| No held-out evaluation at all | "The recommendations look reasonable to me" | Leave-one-out, or a time-based holdout, before shipping anything |

---

## In the real world: the recommendation that didn't need a model

In September 2026, Riverstone's e-commerce contractor proposes a matrix-factorization recommender for the customer web portal, priced at a monthly licence fee. Vikram asks Meera to check whether it's worth it before signing.

Meera builds this chapter's comparison on Riverstone's real order data. Item-based collaborative filtering and a modestly sized ALS model land between 65% and 69% hit rate@5, ahead of the 63% popularity baseline. That's a real, measurable win, until she checks it against a much simpler alternative: **segment-level popularity**, the one from section 42.7, computed with a dozen lines of pandas and no model-fitting at all.

Segment-level popularity actually edges ahead of every model on this catalog: 71.5% hit rate@5, against 68.7% for the best item-based filtering and 67.9% for ALS, and it leads on NDCG too. It costs a fraction of the engineering: no matrix to retrain, no factor count to tune, no cold-start logic needed, because a new customer's segment is known the moment they're onboarded. Her recommendation to Vikram isn't "don't build a recommender"; it's "start with the free version, measure it in production, and only pay for the fancier model once we can show it beats what a dozen lines of code already gets us." Riverstone launches the segment-popularity list first, with a click-through-rate dashboard attached, and Vikram tells the contractor they'll revisit the paid model once there's a baseline number worth beating in production, not just in a backtest.

The lesson isn't that collaborative filtering doesn't work: this chapter's own numbers show it does. It's that Chapter 36's oldest habit (section 36.10), comparing every model against the cheapest baseline that could plausibly work, applies exactly as much to recommenders as it does to lead scoring.

---

## Project: "customers who bought this also bought" for Riverstone

**Goal:** a working recommender, evaluated against at least two baselines, with a plan for cold start.

### Tools you'll need

- **scikit-learn** 1.9.1: `cosine_similarity`, `TruncatedSVD`, `TfidfVectorizer` (reused from Chapter 41).
- **implicit** 0.7.3 (`python -m pip install implicit`, section 42.0): `AlternatingLeastSquares`, purpose-built for implicit-feedback recommenders at scale.
- Not used here but worth knowing: **LightFM** (hybrid recommenders combining content and collaborative signals in one model; its latest release, 1.17 from March 2023, ships no ready-made wheels and often fails to install on current Python versions), **Surprise** (`python -m pip install scikit-surprise`, a research-oriented library for explicit-rating recommenders, less suited to implicit B2B data like Riverstone's).
- The outputs in this chapter were produced on one CPU core with Python 3.11, pandas 3.0.6, NumPy 2.4.6, SciPy 1.17.1, scikit-learn 1.9.1 and implicit 0.7.3, on 29 September 2026.
- **Companion files:** `companion/ch42/products_text.py` adds short descriptions to the Chapter 38 product catalog. Run the chapter's code from `companion/ch42/`, as set up in section 42.0; it reads `../baskets/order_lines.csv`, `../baskets/products.csv`, and `../accounts/accounts.csv`.

**Option A: your own data.** Any customer-by-product (or user-by-item) history: purchases, clicks, views, ratings.

**Option B: Riverstone.** The order lines, products, and accounts data.

**Steps:**

1. **Build the interaction matrix** and report its density.
2. **Popularity baseline**, overall and per segment (or whatever grouping your data has).
3. **Item-based collaborative filtering**, with the item-similarity table for at least one product checked by eye.
4. **One matrix-factorization or implicit-feedback model**, tested at three different factor counts.
5. **Content-based filtering**, if you have any text describing your items; if not, describe why it isn't available and what that costs you at cold start.
6. **Leave-one-out evaluation** of every method above, reporting hit rate@5, NDCG@5 and MRR@5 in one table.
7. **Cold start:** demonstrate what happens with a brand-new item and a brand-new customer under your best method, and propose a fallback for each.
8. **Write a one-page recommendation**: which method (or combination) you'd deploy, what it costs to maintain, and how you'd measure it in production.

**Stretch goals:**

- Try a **time-based** holdout (hide the products in each account's *most recent* order, not a random one) instead of leave-one-out, and see whether the ranking of methods changes.
- Add a **diversity** check: do the top-5 recommendations across all accounts cover most of the catalog, or does a strong model just recommend the same 3 popular items to everyone?
- If LightFM installs on your system, build a **LightFM** hybrid model and compare it directly with this chapter's hand-blended hybrid.
- Measure how evaluation results change as you **shrink** the training data (keep only the most recent 3 months of orders): how much history does a workable recommender actually need?

---

## Recap

- **Popularity** is the recommender baseline every fancier method must beat, and it's identical for every customer.
- **Item-based collaborative filtering** measures product similarity from co-purchase patterns via cosine similarity, with no knowledge of what the products are; **user-based** filtering compares customers instead.
- **Hit rate@k** (the share of customers with a relevant item in the top *k*), **NDCG@k** and **MRR@k**, measured with **leave-one-out**, suit ranked recommendations; NDCG and MRR also reward getting the order right. With one hidden item, recall@k equals the hit rate, and precision@k can't exceed 1 ÷ *k*.
- **Matrix factorization** rebuilds the interaction matrix from two small tables of learned factors, and the scores it puts in empty cells are the recommendations; more factors is not automatically better, especially on small or narrow catalogs: test it.
- **Implicit feedback** (purchases, clicks, quantities) needs confidence-weighted methods, or at least yes/no or log quantities, not rating-prediction methods built for five-star data.
- **Content-based filtering** reuses Chapter 41's TF-IDF and cosine similarity on item descriptions, and is the only method here that works for brand-new items.
- **Cold start** (new products, new customers) needs a fallback: content-based scoring, popularity, or segment membership.
- **Hybrid** methods trade off different failure modes; they aren't automatically better than the best single method, and should be checked, not assumed.
- **Segment-level popularity** is a cheap, robust baseline; on Riverstone's catalog it beats every model on the section 42.7 scoreboard.

---

## Key terms

recommender system · interaction matrix · density · sparsity · popularity baseline · content-based filtering · collaborative filtering · item-based collaborative filtering · user-based collaborative filtering · cosine similarity (recommenders) · hit rate@k · recall@k · precision@k · NDCG (normalized discounted cumulative gain) · discounted cumulative gain (DCG) · ideal DCG · mean reciprocal rank (MRR) · mean average precision (MAP) · leave-one-out evaluation · matrix factorization · latent factors · `TruncatedSVD` · implicit feedback · explicit feedback · confidence weighting · alternating least squares (ALS) · sparse matrix (`csr_matrix`) · cold-start problem · hybrid recommender · segment-level baseline · cross-selling

*(All terms are defined in the Glossary, Appendix A.)*

---

## Check yourself

- [ ] I can explain the difference between content-based and collaborative filtering in one sentence each.
- [ ] I always compare a recommender against a popularity baseline before trusting it.
- [ ] I can compute item-item cosine similarity by hand on a small example.
- [ ] I can say which measure I computed (hit rate@k, recall@k or precision@k), and compute NDCG and reciprocal rank for one list by hand.
- [ ] I evaluate recommenders with leave-one-out or a time-based holdout, not accuracy.
- [ ] I can multiply two small factor tables by hand and point to the recommendation in the result.
- [ ] I know that more matrix-factorization factors isn't automatically better, and I test factor counts on held-out data.
- [ ] I understand why implicit feedback (quantities, clicks) needs different treatment than explicit ratings.
- [ ] I have a plan for cold-start products and cold-start customers before I need one.
- [ ] I check whether a simple, grouped baseline (like segment-level popularity) closes the gap before recommending a complex model.

---

## Exercises

Code exercises run from `companion/ch42/` after the chapter's code (they use `matrix`, `X`, `X_train`, `eligible`, `held_out`, `evaluate`, `item_sim_train`, `content_similarity`, `products`, `accounts`, and the rest). Predict each answer before running it.

### Warm-up

1. *(hand)* Two products' purchase vectors across 4 customers are [2, 0, 4, 0] and [1, 0, 2, 0]. Compute their cosine similarity. What does the result tell you about their relationship?
2. *(hand)* A recommender puts the relevant item at position 3 of a top-5 list. Compute its contribution to DCG (1 ÷ log₂(position + 1), with position counted from 1) and its reciprocal rank. Compare both with the same item at position 1.
3. *(hand)* A catalog has 500 products. A customer has bought 5. What's the matrix's density for that customer's row alone? Why is real-world recommender data usually far sparser than Riverstone's 39%?
4. Which method fits each situation: (a) a product launched yesterday with 0 sales; (b) a returning customer with 200 past orders; (c) a brand-new customer with no history but a known industry; (d) explaining to a manager why two products were bundled?

### Core

5. Compute item-item cosine similarity for **Drum 60L (P17)** instead of Storage Crate 50L. What are its three most similar products, and does the result make business sense?
6. Re-run the leave-one-out evaluation with **k = 3** and **k = 10** instead of 5, for the popularity baseline and item-based CF. How does each method's hit rate change as k grows, and why?
7. Retrain the ALS model with **regularization** set to 0.01 and to 1.0 (instead of 0.1), keeping factors at 8. Report the hit rate@5 for each. What does regularization appear to be doing here?
8. Section 42.7 computed popularity per segment. Now compute **item-based collaborative filtering per segment**: build each segment's product similarity table from its own accounts in `X_train`, score each account with its segment's table, and evaluate it. Does it beat plain item-based CF, and segment-level popularity?
9. For the content-based recommender, find the pair of **different** products with the highest description similarity, and read both descriptions. Does the similarity reflect something a customer would actually care about?
10. Combine collaborative and content scores with weights 0.8/0.2 and 0.2/0.8 instead of 0.5/0.5. Which weighting performs best on hit rate@5, and does either beat item-based CF alone?

### Stretch

11. Implement a simple **"customers who bought this also bought"** function that, given one product ID, returns the top 3 other products by co-occurrence count (not cosine similarity). Compare its output for Storage Crate 50L with section 42.2's cosine-similarity result. Where do they agree and disagree?
12. Evaluate popularity, item-based CF and ALS with 8 factors using a **time-based** holdout: for each account, hide every product in its chronologically last order (by `order_date`) that it hadn't bought in an earlier order, train on the earlier orders only, and count a hit when any hidden product is in the top 5. Do the rankings of methods change?
13. *(Optional: LightFM may not install.)* LightFM's latest release, 1.17, dates from March 2023 and has no ready-made wheels, so `python -m pip install lightfm` has to compile it, and on current Python versions that often fails. If it installs for you, build a tiny LightFM model using both the interaction matrix and item descriptions as features, and compare its hit rate@5 with this chapter's hand-built hybrid. If the install fails, skip this exercise.

### Think about it

14. A colleague says: "Our recommender has a 70% hit rate@5, so it must be great." What two questions would you ask before agreeing?
15. Explain to a non-technical account manager, in three sentences, why a brand-new product can't be recommended by collaborative filtering no matter how good the algorithm is.
16. A hybrid recommender scores worse than its best single component on your evaluation. A colleague argues you should keep the hybrid anyway "because it's more sophisticated." How do you respond?

---

## Answers

*(Every calculation was checked, and every code output shown is real.)*

**1.** Dot product = 2×1 + 0×0 + 4×2 + 0×0 = 2 + 8 = **10**. Lengths: √(2² + 4²) = √20 = 4.472; √(1² + 2²) = √5 = 2.236. Cosine = 10 ÷ (4.472 × 2.236) = 10 ÷ 10.0 = **1.0**. A cosine of exactly 1 means the two vectors point in exactly the same direction: the second product is bought in exactly half the quantity of the first by every customer, a perfectly proportional relationship, even though the raw counts differ.

**2.** At position 3, the DCG contribution = 1 ÷ log₂(3 + 1) = 1 ÷ log₂(4) = 1 ÷ 2 = **0.5**, and the reciprocal rank = 1 ÷ 3 = **0.333**. At position 1, both are 1 ÷ log₂(2) = 1 ÷ 1 = **1.0** and 1 ÷ 1 = **1.0**. The same relevant item is worth twice as much at position 1 on NDCG, and three times as much on reciprocal rank: both are built to reward moving a good recommendation higher up the list, not just including it somewhere.

**3.** Density for that one row = 5 ÷ 500 = **1%**. Riverstone's overall 39% density comes from having only 24 products in total; real catalogs with thousands or millions of items give any single customer's row a density far below 1%, because no customer can plausibly buy even a small fraction of everything on offer. That's why real-world recommender systems are built around sparse matrices and methods (like matrix factorization) designed to handle mostly-empty data.

**4.** (a) **Content-based filtering**: a zero-sales product has no collaborative signal at all, only a description (section 42.6). (b) **Item-based collaborative filtering or matrix factorization**: plenty of history to learn from directly. (c) **Popularity baseline, ideally segmented by known attributes** (industry, company size): no purchase history exists yet, but non-purchase information can substitute (sections 42.6 and 42.7). (d) **Content-based similarity, or the raw co-occurrence counts from market basket analysis (Chapter 38)**: both give a human-readable reason ("similar description" or "bought together X% of the time") that a collaborative-filtering score alone doesn't.

**5.**

In [34]:
drum_id = "P17"
top_similar_drum = sim_table[drum_id].sort_values(ascending=False).head(3)
print(f"products bought most like {names[drum_id]!r}:")
for product_id, score in top_similar_drum.items():
    print(f"  {names[product_id]:<22} similarity {score:.3f}")

products bought most like 'Drum 60L':
  Drum Tap Fitting       similarity 0.785
  Dolly Wheels (set)     similarity 0.722
  Pallet Box             similarity 0.697


This is section 42.2's code with `P17` as the target and three matches instead of four.

The top match is the Drum Tap Fitting, the drum's own accessory and one of the pairs Chapter 38's rules found. The next matches, Dolly Wheels and the Pallet Box, are other industrial items bought by the same wholesale-heavy customers. This makes clear business sense: drums are an industrial product, bought by industrial buyers, who also buy other industrial products, and the algorithm arrives there with no idea what a "drum" or a "fitting" actually is.

**6.**

In [35]:
for k in [3, 5, 10]:
    hit_pop_k, _, _ = evaluate(lambda i: pop_counts, k=k)
    hit_cf_k, _, _ = evaluate(item_cf_score, k=k)
    print(f"k={k:>2}:  popularity {hit_pop_k:.1%}   item-based CF {hit_cf_k:.1%}")

k= 3:  popularity 45.8%   item-based CF 52.5%


k= 5:  popularity 62.7%   item-based CF 65.0%


k=10:  popularity 85.6%   item-based CF 83.7%


The hit rate rises for both methods as *k* grows, because a longer list has more chances to contain the one hidden product. At the extreme, recommending every product the account doesn't own guarantees a hit (hit rate 100%), which is why *k* must always be stated. Notice that at k = 10, popularity (85.6%) actually edges ahead of item-based CF (83.7%): with a 24-product catalog, a top-10 list already covers 10 ÷ 24 = 42% of everything on offer, so the two methods' lists overlap heavily and the gap that mattered at k = 3 and 5 disappears. That's why a hit rate must be reported *with* the k it was measured at, chosen to reflect a realistic list length, and why NDCG (which discounts by position rather than just checking presence) is the more informative single number when comparing methods.

**7.**

In [ ]:
default_hit = als_results[8][0]
print(f"regularization=0.1 (chapter default): hit rate@5 {default_hit:.1%}")
for reg in [0.01, 1.0]:
    reg_model = implicit.als.AlternatingLeastSquares(
        factors=8, regularization=reg, iterations=20, random_state=42
    )
    reg_model.fit(confidence, show_progress=False)
    reg_hit, _, _ = evaluate(
        lambda i, m=reg_model: m.user_factors[i] @ m.item_factors.T
    )
    print(f"regularization={reg:<5} hit rate@5 {reg_hit:.1%}")

> **Not run here.** This cell needs something this machine does not have: a database, an API key, a running service, or command-line arguments. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
regularization=0.1 (chapter default): hit rate@5 67.9%
regularization=0.01  hit rate@5 68.0%
regularization=1.0   hit rate@5 67.6%
```

`als_results[8][0]` is the hit rate stored for 8 factors in section 42.4; the loop fits two more models the same way, changing only `regularization`.

Regularization penalizes large factor values, the matrix-factorization equivalent of Chapter 37's ridge penalty (section 37.2). Here the three settings land within half a point of each other (67.6%, 67.9%, 68.0%): regularization isn't the lever that matters on this catalog. The factor-count experiment in section 42.4 showed a much larger swing, from 67.9% at 8 factors down to 46.0% at 16. The number of factors, not the regularization strength, is what needs care here, a reminder that not every hyperparameter matters equally for a given dataset, and testing is how you find out which ones do.

**8.**

In [37]:
segment_similarity = {}
for segment in ["Retail", "Hospitality", "Wholesale"]:
    sim = cosine_similarity(X_train[account_segments == segment].T)
    np.fill_diagonal(sim, 0)
    segment_similarity[segment] = sim


def segment_cf_score(i):
    return X_train[i] @ segment_similarity[account_segments[i]]


hit_seg_cf, ndcg_seg_cf, mrr_seg_cf = evaluate(segment_cf_score)
print(f"item CF per segment:      hit rate@5 {hit_seg_cf:.1%}   NDCG@5 {ndcg_seg_cf:.3f}")
print(f"item CF, all accounts:    hit rate@5 {hit_cf:.1%}   NDCG@5 {ndcg_cf:.3f}")
print(f"segment-level popularity: hit rate@5 {hit_seg:.1%}   NDCG@5 {ndcg_seg:.3f}")

item CF per segment:      hit rate@5 72.1%   NDCG@5 0.540
item CF, all accounts:    hit rate@5 65.0%   NDCG@5 0.495
segment-level popularity: hit rate@5 71.5%   NDCG@5 0.533


- `X_train[account_segments == segment]` keeps one segment's rows, and `.T` compares its products, as in section 42.2; each segment gets its own 24 × 24 table.
- `segment_cf_score` looks up the account's segment and uses that segment's table.

Item-based filtering computed within each segment reaches 72.1% hit rate@5 and NDCG 0.540, far above the same method on all accounts together (65.0%, 0.495) and a little above segment-level popularity (71.5%, 0.533). Similarities learned from similar customers are more relevant: in a wholesale-only table, the Drum 60L's neighbors are what wholesale accounts buy with it, not what the whole customer base does. But the gain over segment-level popularity is under a point, for a model that needs three similarity tables kept up to date. Whether that's worth it is a business call, and now it can be made with numbers.

**9.** `content_similarity` is a NumPy array, so copy it, zero the diagonal (a product with itself), and only then wrap it in a DataFrame:

In [38]:
no_diagonal = content_similarity.copy()
np.fill_diagonal(no_diagonal, 0)
content_no_diagonal = pd.DataFrame(no_diagonal, index=items, columns=items)
best_pair = content_no_diagonal.stack().idxmax()
print(
    f"most similar different products: {names[best_pair[0]]!r} and "
    f"{names[best_pair[1]]!r}, similarity {content_no_diagonal.loc[best_pair]:.3f}"
)
for product_id in best_pair:
    print(f"  {product_id}: {descriptions[product_id]}")

most similar different products: 'Food Container 2L' and 'Food Container 5L', similarity 1.000
  P07: 2 litre airtight food storage container, microwave and freezer safe, BPA free plastic.
  P08: 5 litre airtight food storage container, microwave and freezer safe, BPA free plastic.


- `.stack()` turns the 24 × 24 table into one long Series indexed by (row, column) pairs, and `.idxmax()` returns the pair with the largest value.
- Zeroing the diagonal *before* building the DataFrame matters: with pandas' copy-on-write, `DataFrame.values` can hand back a read-only array, and `np.fill_diagonal` on it stops with `ValueError: underlying array is read-only`.

The top pair is the 2-litre and 5-litre Food Containers, with a similarity of exactly 1.000. Their descriptions differ only in the size, "2" against "5", and `TfidfVectorizer` by default ignores one-character tokens, so to it the two texts are identical. A customer would see them as two sizes of the same product: a sensible alternative to each other, not an add-on. This is section 42.5's watch-out in practice: TF-IDF similarity reflects shared *vocabulary*, and a size, a colour or a model number that matters to the buyer can vanish from the text the model sees. Read a few flagged pairs by hand before trusting the scores.

**10.**

In [39]:
for cf_weight in [0.8, 0.2]:
    hit_w, ndcg_w, _ = evaluate(lambda i, w=cf_weight: hybrid_score(i, weight=w))
    print(
        f"weight {cf_weight} collaborative / {1 - cf_weight:.1f} content: "
        f"hit rate@5 {hit_w:.1%}   NDCG@5 {ndcg_w:.3f}"
    )
print(f"item-based CF alone: hit rate@5 {hit_cf:.1%}")

weight 0.8 collaborative / 0.2 content: hit rate@5 65.1%   NDCG@5 0.486


weight 0.2 collaborative / 0.8 content: hit rate@5 58.7%   NDCG@5 0.397
item-based CF alone: hit rate@5 65.0%


Weighting heavily toward collaborative filtering (0.8) lands level with item-based CF on hit rate (65.1% against 65.0%, a difference of a handful of accounts) but below it on NDCG (0.486 against 0.495); weighting toward content (0.2 collaborative) drags it down toward content-based's weaker standalone score (58.7%). Neither weighting clearly beats item-based CF alone on this catalog: a clean illustration that a hybrid is a tool for handling cases the best single method can't reach (like cold start), not a guaranteed accuracy improvement over that method.

**11.**

In [40]:
def bought_together(product_id, top_n=3):
    orders_with_product = order_baskets[
        order_baskets.apply(lambda s: product_id in s)
    ]
    counts = pd.Series(
        [
            p
            for basket in orders_with_product
            for p in sorted(basket)
            if p != product_id
        ]
    ).value_counts()
    return counts.head(top_n)


print("co-occurrence count method:")
print(bought_together("P01").rename(index=names).to_string())
print("\ncosine similarity method (section 42.2):")
cosine_top = sim_table["P01"].sort_values(ascending=False).head(3)
print(cosine_top.rename(index=names).round(3).to_string())

co-occurrence count method:
Crate Lid (50L)       2800
Airtight Seal Pack     711
Crate Lid (80L)        711

cosine similarity method (section 42.2):
Crate Lid (50L)     0.647
Crate Lid (80L)     0.573
Drum Tap Fitting    0.555


`bought_together` repeats section 42.7's co-occurrence count for any product; `top_n=3` is a default value, so `bought_together("P01", top_n=5)` would return five. `order_baskets` is section 42.7's Series of baskets.

Both methods put the matching lid first, in 2,800 orders, the same count Chapter 38, section 38.9 found. Further down they disagree. Raw co-occurrence counts favor whatever is simply *popular*: the Airtight Seal Pack, the most-bought product of all, ties for second with Crate Lid (80L) at 711 orders. Cosine similarity divides by each product's overall size (its vector length, Chapter 35), so a product that is everywhere doesn't win just by being everywhere; it ranks Crate Lid (80L) and the Drum Tap Fitting next. Note the two also count differently: co-occurrence counts **orders**, while section 42.2's cosine uses each **account's** total quantities, so some of the disagreement comes from that, not only from popularity.

**12.** First, find each account's last order, and build a training matrix from the earlier orders only:

In [41]:
by_date = lines.sort_values(["order_date", "order_id"])
last_order = by_date.groupby("account_id")["order_id"].last()
is_last = lines["order_id"].isin(last_order)
earlier = (
    lines[~is_last]
    .groupby(["account_id", "product_id"])["quantity"].sum()
    .unstack(fill_value=0)
    .reindex(index=matrix.index, columns=items, fill_value=0)
)
X_before = earlier.to_numpy().astype(float)
new_in_last = (
    lines[is_last]
    .groupby("account_id")["product_id"].apply(set)
    .reindex(matrix.index)
)
time_targets = {}
for i, account_id in enumerate(matrix.index):
    in_last = new_in_last[account_id]
    if isinstance(in_last, set) and X_before[i].sum() > 0:
        columns = [items.index(p) for p in sorted(in_last)]
        targets = [j for j in columns if X_before[i, j] == 0]
        if targets:
            time_targets[i] = targets
print(f"{len(time_targets):,} accounts have earlier orders and a new product last time")

2,217 accounts have earlier orders and a new product last time


- `sort_values(["order_date", "order_id"])` puts the lines in time order (the order ID breaks ties on the same day), so `groupby(...).last()` gives each account's latest order ID.
- `earlier` is the interaction matrix built from every order *except* the last; `unstack(fill_value=0)` is a shortcut for the pivot-and-fill of section 42.1, and `reindex` puts the rows and columns in the same order as `matrix`.
- `new_in_last` holds each account's last order as a set of product IDs. `isinstance(in_last, set)` checks that the account has one (an account with a single order has nothing earlier to learn from, and `X_before[i].sum() > 0` checks for that too).
- `time_targets` keeps, for each account with some history, the column positions of the products in its last order that it had never bought before: the ones a recommender could have suggested.

Then score three methods, each trained on `X_before` only:

In [ ]:
def time_evaluate(score_fn, k=5):
    hits = 0
    for i, targets in time_targets.items():
        scores = score_fn(i).copy()
        scores[X_before[i] > 0] = -np.inf
        top_k = np.argsort(-scores)[:k]
        hits += any(t in top_k for t in targets)
    return hits / len(time_targets)


before_pop = (X_before > 0).sum(axis=0).astype(float)
before_sim = cosine_similarity(X_before.T)
np.fill_diagonal(before_sim, 0)
before_als = implicit.als.AlternatingLeastSquares(
    factors=8, regularization=0.1, iterations=20, random_state=42
)
before_als.fit(csr_matrix(np.log1p(X_before)), show_progress=False)
user_f, item_f = before_als.user_factors, before_als.item_factors
print(f"popularity     hit rate@5 {time_evaluate(lambda i: before_pop):.1%}")
print(f"item-based CF  hit rate@5 {time_evaluate(lambda i: X_before[i] @ before_sim):.1%}")
print(f"ALS, 8 factors hit rate@5 {time_evaluate(lambda i: user_f[i] @ item_f.T):.1%}")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
popularity     hit rate@5 70.9%
item-based CF  hit rate@5 67.0%
ALS, 8 factors hit rate@5 62.7%
```

- `time_evaluate` is `evaluate` with two changes: it excludes what the account bought in `X_before`, and it counts a hit when any of the account's hidden products makes the top 5. `any(t in top_k for t in targets)` is True when at least one did; adding True counts as 1.
- `before_pop`, `before_sim` and `before_als` are the popularity counts, the item similarities and the ALS model rebuilt from `X_before`, exactly as in sections 42.3 and 42.4. `user_f` and `item_f` are short names for the model's two factor tables, to keep the last line short.

Popularity scores 70.9%, item-based CF 67.0% and ALS 62.7%: under a time-based holdout the ranking **changes**, and the plain popularity baseline comes first. The numbers aren't directly comparable with leave-one-out's (different accounts, and some have more than one hidden product), but the order is the finding. When the question is "what new product will this account add next?", the products most accounts buy were a better guess than anything learned from the account's own mix. A time-based holdout is the more realistic test for any recommender that will be used to predict *future* purchases, and this answer is Chapter 36's lesson (section 36.3) again: a random split can flatter a model in ways a time-respecting split won't. Before choosing a model for production, test it the way it will be used.

**13.** *(No worked output: LightFM would not install on the Python version used for this book, so this exercise is optional.)* LightFM builds one model that blends interaction data and item features (like this chapter's descriptions) directly, rather than combining two separately trained models by hand as section 42.5's hybrid does. If you can install it, compare its hit rate@5 against this chapter's hybrid using the same leave-one-out procedure from section 42.3, and note that LightFM's real advantage over a hand-blended hybrid tends to show up on colder items and customers, not necessarily on the well-populated ones this catalog mostly contains.

**14.** First: *"70% hit rate@5 compared with what?"* A popularity baseline, or a segment-level baseline, might already reach 60% or more (on Riverstone's catalog they reach 63% and 72%), and the real value added by the fancier model could be a few points, not seventy. Second: *"Is that measured with a proper held-out test (leave-one-out or time-based), or just eyeballed on training data?"* A recommender that was never evaluated against unseen data can look impressive and still be memorizing rather than generalizing, Chapter 37's overfitting lesson (section 37.10) applied to ranking. A third question worth asking: *"Is it really precision?"* With one hidden item, precision@5 can't exceed 20%, so a 70% figure is a hit rate.

**15.** "Collaborative filtering works entirely by comparing what different customers have bought: it has no idea what a product actually is, only which customers bought it. A brand-new product has never been bought by anyone, so there's nothing in the purchase data to compare it against, no matter how sophisticated the algorithm is. The only way to recommend it before it has sales history is to use what we know about the product itself, like its description or category, which is a completely different method (content-based filtering)."

**16.** Ask what the hybrid is *for*. If the goal is the single best hit rate today, on a catalog and customer base this well populated, the evidence says drop the hybrid and use the better single method. But if new products or new customers are a regular occurrence, which they are for most real catalogs, the hybrid's value shows up specifically in those cold-start cases that a leave-one-out test on existing, well-populated accounts doesn't measure at all. The right response isn't "sophistication is its own reward," it's "let's measure the hybrid specifically on cold-start cases before deciding whether it earns its extra complexity."

---

## Where this leads

- **Chapter 38, Unsupervised Learning,** supplied the basket data and the market-basket rules this chapter turned into personalized rankings.
- **Chapter 41, NLP Foundations,** supplied the TF-IDF and cosine-similarity machinery behind content-based filtering, unchanged.
- **Chapter 39, Evaluation, Tuning, Interpretation & Honesty,** governs how you'd defend a recommender's business value the same way it governs any other model.
- **Chapter 48, Big Data & Distributed Compute,** covers the engineering for data too large for one machine, which a catalog of millions of items and customers quickly becomes.
- **Chapter 54, Generative AI & Large Language Models,** revisits embeddings at a much larger scale, including recommending "similar products" by embedding similarity as one of their standard uses.
- **Interview preparation:** collaborative versus content-based filtering, the cold-start problem, and hit rate versus NDCG are common questions for applied ML roles; exercises 14 to 16 above are good practice, alongside the Machine Learning Question Bank (Chapter 74).